In [ ]:
# Parameters (tagged "parameters": a pipeline or the job API can override any of them)
seed = None                       # None = seed recorded in the mirrored ERP run (bronze.ref_synthetic_run)
dq_defect_rate = None             # share of rows that also get a planted DQ defect; None = mirrored run value
engagement_scale = 1.0            # multiplies app opens, wishlist adds and launch reminders
members_per_location = 36000      # loyalty members served per store / e-commerce node (drives active_members)
hero_small_return_share = 0.25    # share of hero-style returns forced to SIZE_TOO_SMALL (the hero last runs small)
weather_missing_rate = 0.012      # DQ-07 market-days with no observation at all
weather_null_rate = 0.013         # DQ-07 observations delivered with null temperatures
weather_local_time_market = "BR"  # DQ-08 weather provider that sends local time instead of UTC
event_v2_rate = 0.6               # DQ-01 share of events re-published as version 2
hero_style = "HM9443"             # launch style the demo story is about
spike_market = "JP"               # market whose event drives a short engagement spike
spike_event = "Tokyo City Run Expo"
slow_market = "GB"                # market that receives the DQ-05 bulk return (returns > sales)
bronze_schema = "bronze"
dbx_sql_source = "lh_databricks.dbo"   # SQL endpoint for the catalog-managed Databricks mirror tables

# 01 - Bronze: generate the SYNTHETIC file-feed sources (DS-11..DS-14)

UC1 *Athletic Marketplace Inventory Rebalancing* (Customized_Human_Centered_Agentic_Use_Cases_6_Slides.pptx, slides 1-2).
Kinetiq is a fictional athletic brand. **Every row is SYNTHETIC** (`record_label = 'SYNTHETIC'`).

The Databricks (DS-01/02) and ERP/WMS (DS-03..10) sources are mirrored into Fabric and appear in Bronze as zero-copy
shortcuts. The four feeds below have no system of record in this demo, so this notebook generates them directly
into Bronze Delta tables: **no landing files, no custom library**. Each table has its own cell. The generators read the
mirrored masters and sell-through so every SKU, location, market and date matches the mirrors:

| Cell | Bronze table | Requirement | Driven by | Planted defects |
|---|---|---|---|---|
| Weather | `file_weather_daily` | DS-13 | climate normals per flagship city | missing days, null temperatures (DQ-07); local time from one provider (DQ-08) |
| Regional events | `file_regional_events` | DS-14 | ERP `regional_event` calendar | superseded v1 rows (DQ-01) |
| Member engagement | `file_member_engagement` | DS-11 | daily mirrored sell-through per style-colour, pre-launch wishlists, event spike | re-sent rows (DQ-01), unknown colour (DQ-02) |
| Returns | `file_returns` | DS-12 | `units_returned` in the mirrored sell-through | re-exports (DQ-01), unknown SKU (DQ-02), bulk return > sales (DQ-05), store local time vs e-commerce UTC (DQ-08) |

Every column is written as a string (schema-on-read, as a vendor feed would arrive) plus lineage columns
`_ingest_ts`, `_source_system = 'synthetic_generator'`, `_source_object`. Silver does the typing and cleansing.
Generation is deterministic: each table uses its own RNG stream from `seed`, so the same mirrors and seed give identical rows.

In [ ]:
import json, math, zlib
from datetime import date, datetime, time, timedelta, timezone
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd

SYNTHETIC = "SYNTHETIC"


def rng_for(seed, *key):
    """Independent RNG stream per table / key, so cells can be rerun in any order with identical output."""
    return np.random.default_rng([int(seed), zlib.crc32("|".join(map(str, key)).encode())])


def utc_stamp(days, plus_days, hours, minutes):
    """ISO-8601 UTC text for each date in `days`, shifted by plus_days, at hh:mm (export / publish timestamps)."""
    t = pd.to_datetime(pd.Series(list(days)).astype(str)) + pd.Timedelta(days=plus_days, hours=hours, minutes=minutes)
    return t.dt.strftime("%Y-%m-%dT%H:%M:%SZ").to_numpy()

## Read the mirrored inputs (zero-copy Bronze shortcuts)

In [ ]:
import com.microsoft.spark.fabric   # registers spark.read.synapsesql for the catalog-managed Databricks mirror
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")

def bronze(name):
    return spark.table(f"{bronze_schema}.{name}").toPandas()

m_run, m_market, m_location = bronze("ref_synthetic_run"), bronze("ref_market"), bronze("erp_location")
m_sku, m_style, m_price, m_event = bronze("erp_product_sku"), bronze("erp_product_style"), bronze("erp_price"), bronze("erp_regional_event")
m_sales = (spark.read.synapsesql(f"{dbx_sql_source}.sell_through_daily")
           .select("business_date", "location_id", "channel", "style_id", "color", "size",
                   "units_sold", "units_returned", "_extracted_at")
           .toPandas())
print(f"sell-through rows read: {len(m_sales):,}")

generated = {}


def save_bronze(name, pdf):
    """Write one generated table to bronze.file_<name>: every value as text plus lineage columns."""
    pdf = pdf.astype(object).where(pdf.notna(), None)
    for c in pdf.columns:
        pdf[c] = pdf[c].map(lambda v: None if v is None else str(v))
    sdf = (spark.createDataFrame(pdf, StructType([StructField(c, StringType(), True) for c in pdf.columns]))
           .withColumn("_ingest_ts", F.current_timestamp())
           .withColumn("_source_system", F.lit("synthetic_generator"))
           .withColumn("_source_object", F.lit(f"01_bronze_generate_sources/{name}")))
    table = f"{bronze_schema}.file_{name}"
    sdf.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(table)
    generated[name] = pdf
    print(f"{table}: {spark.table(table).count():,} rows")

In [ ]:
# Align with the mirrored world: run parameters, masters and de-duplicated daily sell-through
run = json.loads(m_run.params_json.iloc[0])
seed = int(seed if seed is not None else run["seed"])
dq_rate = float(dq_defect_rate if dq_defect_rate is not None else run["dq_defect_rate"])
as_of = date.fromisoformat(str(run["as_of"])[:10])

markets = m_market.set_index("market_id")
loc = m_location[["location_id", "location_type", "market_id", "city", "timezone"]].astype(str)
flagship = loc[loc.location_type == "STORE"].sort_values("location_id").groupby("market_id").first()
skus = m_sku[["sku_id", "style_id", "color_code", "size"]].astype(str)
launch = pd.to_datetime(m_style.set_index("style_id").launch_date).dt.date
price = pd.to_numeric(m_price.set_index(["market_id", "style_id"]).full_price_local).astype(float)
events = m_event.assign(start_date=pd.to_datetime(m_event.start_date).dt.date,
                        end_date=pd.to_datetime(m_event.end_date).dt.date).sort_values("event_id")

s = m_sales.copy()
s["business_date"] = pd.to_datetime(s.business_date).dt.date
s["_extracted_at"] = pd.to_datetime(s["_extracted_at"].astype(str), utc=True, errors="coerce")
for c in ("units_sold", "units_returned"):
    s[c] = pd.to_numeric(s[c], errors="coerce")
s["size"] = s["size"].astype(str)
s = s.sort_values("_extracted_at", kind="stable").drop_duplicates(            # restated / re-sent extracts: latest wins
    ["business_date", "location_id", "style_id", "color", "size"], keep="last")
s["sku_id"] = s.style_id + "-" + s.color + "-" + s["size"]
s = s[s.sku_id.isin(skus.sku_id) & (s.units_sold >= 0) & (s.units_returned.fillna(0) >= 0)]
s = s.merge(loc, on="location_id")                                             # drops unknown locations
s = s.sort_values(["business_date", "location_id", "sku_id"]).reset_index(drop=True)
start, end = s.business_date.min(), as_of - timedelta(days=1)
print(f"seed={seed} dq_rate={dq_rate} as_of={as_of} window={start}..{end} sales rows={len(s):,} markets={sorted(s.market_id.unique())}")

## Tables

In [ ]:
# file_weather_daily (DS-13): one observation per market flagship city per day
# Monthly climate normals (mean temp C, mean precip mm/day, Jan..Dec): rounded public climatology; observations are synthetic.
CLIMATE = {
    "US": ([0.5, 1.9, 5.8, 11.9, 17.5, 22.6, 25.5, 24.9, 21.0, 14.9, 9.1, 3.8], [2.9, 2.8, 3.6, 3.4, 3.2, 3.6, 3.7, 3.5, 3.5, 3.5, 3.0, 3.3]),
    "GB": ([5.2, 5.3, 7.6, 9.9, 13.3, 16.5, 18.7, 18.5, 15.7, 12.0, 8.0, 5.5], [1.8, 1.4, 1.3, 1.5, 1.6, 1.5, 1.5, 1.6, 1.6, 2.2, 2.0, 1.8]),
    "JP": ([5.4, 6.1, 9.4, 14.3, 18.8, 21.9, 25.7, 26.9, 23.3, 18.0, 12.5, 7.7], [1.7, 2.0, 3.8, 4.1, 4.5, 5.6, 5.0, 5.0, 7.0, 6.3, 3.2, 1.7]),
    "BR": ([22.9, 23.2, 22.6, 20.7, 18.6, 17.4, 17.0, 18.0, 19.0, 20.0, 21.0, 22.2], [7.7, 7.0, 5.1, 2.6, 2.4, 1.7, 1.4, 1.1, 2.7, 4.1, 4.8, 6.5]),
    "AU": ([23.5, 23.5, 22.4, 19.8, 16.9, 14.5, 13.6, 14.6, 16.9, 19.0, 20.6, 22.4], [3.3, 4.2, 4.2, 4.2, 3.9, 4.0, 2.6, 2.6, 2.0, 2.5, 2.8, 2.5]),
    "DE": ([0.6, 1.4, 4.8, 9.6, 14.5, 17.8, 19.9, 19.5, 15.1, 10.2, 5.1, 1.8], [1.4, 1.2, 1.3, 1.3, 1.8, 1.9, 2.0, 1.9, 1.6, 1.2, 1.4, 1.5]),
    "FR": ([5.0, 5.6, 8.8, 11.6, 15.2, 18.6, 20.6, 20.4, 16.9, 13.0, 8.3, 5.5], [1.6, 1.6, 1.6, 1.8, 2.1, 1.7, 2.0, 1.7, 1.6, 2.0, 1.7, 1.9]),
    "CA": ([-5.5, -4.5, 0.0, 7.0, 13.5, 19.0, 22.0, 21.0, 16.5, 9.5, 3.5, -2.0], [2.0, 1.8, 1.8, 2.3, 2.6, 2.4, 2.4, 2.6, 2.5, 2.1, 2.5, 2.0]),
    "CN": ([4.2, 6.0, 9.7, 15.3, 20.5, 24.3, 28.6, 28.3, 24.6, 19.4, 13.5, 7.0], [2.4, 2.1, 3.4, 3.5, 3.6, 6.1, 4.8, 6.6, 4.6, 2.0, 1.7, 1.4]),
    "KR": ([-2.4, 0.4, 5.7, 12.5, 17.8, 22.2, 24.9, 25.7, 21.2, 14.8, 7.2, 0.4], [0.6, 0.9, 1.5, 2.4, 3.4, 4.9, 13.1, 11.6, 4.6, 1.7, 1.7, 0.7]),
}
TEMP_SD = 2.6   # daily anomaly std-dev (C), AR(1) with phi 0.7 so warm / cold spells last a few days

rows = []
for m in sorted(set(flagship.index) & set(CLIMATE)):
    r = rng_for(seed, "ds13", m)
    temps, precs = CLIMATE[m]
    city, tz = flagship.loc[m, "city"], ZoneInfo(markets.loc[m, "timezone"])
    local_provider = m == weather_local_time_market
    anom, d = 0.0, start
    while d <= end:
        mi = d.month - 1
        normal = temps[mi] + (temps[(mi + 1) % 12] - temps[mi]) * max(0.0, (d.day - 1) / 30.0 - 0.5)
        anom = 0.7 * anom + r.normal(0, TEMP_SD * math.sqrt(1 - 0.49))
        avg, spread = normal + anom, r.uniform(5, 10)
        p_wet = max(0.2, min(0.85, precs[mi] / 6 + 0.15))
        precip = float(np.round(r.gamma(0.9, precs[mi] / 0.9 / p_wet), 1)) if r.random() < min(0.85, precs[mi] / 6 + 0.15) else 0.0
        cond = "STORM" if precip > 25 else "RAIN" if precip > 2 else "DRIZZLE" if precip > 0 else "CLEAR" if r.random() < 0.55 else "CLOUDY"
        cond = "SNOW" if avg < 1 and precip > 0 else cond
        obs_local = datetime.combine(d, time(23, 0))
        observed_at = (obs_local.strftime("%Y-%m-%d %H:%M:%S") if local_provider
                       else obs_local.replace(tzinfo=tz).astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"))
        row = dict(business_date=d.isoformat(), market_id=m, city=city, station_id=f"WX-{m}-{city[:3].upper()}",
                   temp_max_c=f"{avg + spread / 2:.1f}", temp_min_c=f"{avg - spread / 2:.1f}", temp_avg_c=f"{avg:.1f}",
                   temp_normal_c=f"{normal:.1f}", precip_mm=f"{precip:.1f}", precip_normal_mm=f"{precs[mi]:.1f}",
                   condition=cond, observed_at=observed_at,
                   provider="SynthClima " + m if local_provider else "SynthWeather Global")
        u = r.random()
        if u >= weather_missing_rate:                                       # DQ-07: some market-days never arrive
            if u < weather_missing_rate + weather_null_rate:                # DQ-07: observation with null temperatures
                row.update(temp_max_c=None, temp_min_c=None, temp_avg_c=None)
            rows.append(row)
        d += timedelta(days=1)

weather = pd.DataFrame(rows).assign(record_label=SYNTHETIC)
save_bronze("weather_daily", weather)

In [ ]:
# file_regional_events (DS-14): external events calendar built from the ERP calendar, re-published as v2 (DQ-01)
EVENT_TYPES = (("Marathon", "MARATHON"), ("Run", "RUNNING_EVENT"), ("Expo", "RUNNING_EVENT"),
               ("Season", "SCHOOL_SPORTS_SEASON"), ("Friday", "RETAIL_HOLIDAY"), ("Boxing", "RETAIL_HOLIDAY"),
               ("Soldes", "RETAIL_HOLIDAY"), ("Singles", "RETAIL_HOLIDAY"), ("Week", "PUBLIC_HOLIDAY"),
               ("Carnaval", "CULTURAL_FESTIVAL"), ("Chuseok", "PUBLIC_HOLIDAY"))

r = rng_for(seed, "ds14")
rows = []
for e in events.itertuples():
    etype = next((t for k, t in EVENT_TYPES if k.lower() in e.event_name.lower()), "OTHER")
    base = int(r.integers(8, 60)) * 1000 if etype in ("MARATHON", "RUNNING_EVENT") else int(r.integers(200, 900)) * 1000
    pub = e.start_date - timedelta(days=int(r.integers(45, 120)))
    rec = dict(event_id=e.event_id, market_id=e.market_id, event_name=e.event_name, event_type=etype,
               city=flagship.city.get(e.market_id), start_date=e.start_date.isoformat(), end_date=e.end_date.isoformat(),
               categories=e.categories, expected_attendance=base, demand_uplift_hint=float(e.demand_uplift),
               source="SynthEvents Calendar API", version=1, published_at=f"{pub.isoformat()}T09:00:00Z")
    rows.append(rec)
    v2 = pub + timedelta(days=20)
    if r.random() < event_v2_rate and v2 < as_of:                             # DQ-01: updated attendance, keep latest
        rows.append({**rec, "version": 2, "expected_attendance": int(base * r.uniform(1.05, 1.3)),
                     "published_at": f"{v2.isoformat()}T09:00:00Z"})

regional_events = pd.DataFrame(rows).assign(record_label=SYNTHETIC)
save_bronze("regional_events", regional_events)

In [ ]:
# file_member_engagement (DS-11): loyalty-app engagement per day x market x style-colour
# Expected demand = mirrored units sold that day. Members wishlist styles in the 10 days before launch,
# and the hero style gets an engagement spike in the spike market during the spike event.
PRE_LAUNCH_DAYS, EVENT_MULT = 10, 2.4
key = ["market_id", "style_id", "color"]

d = s.groupby(["business_date"] + key, as_index=False).units_sold.sum().rename(columns={"units_sold": "exp_units"})
d["pre_launch"] = False
pre = []
for (mk, st, co), g in d.groupby(key):
    L = launch.get(st)
    if L is None or pd.isna(L) or L - timedelta(days=PRE_LAUNCH_DAYS) < start or L > end:
        continue
    base = g[g.business_date < L + timedelta(days=14)].exp_units.mean()
    pre += [dict(business_date=L - timedelta(days=k), market_id=mk, style_id=st, color=co, exp_units=base, pre_launch=True)
            for k in range(PRE_LAUNCH_DAYS, 0, -1)]
d = pd.concat([d, pd.DataFrame(pre, columns=d.columns)], ignore_index=True).sort_values(["business_date"] + key).reset_index(drop=True)

ev = events[(events.market_id == spike_market) & (events.event_name == spike_event)]
ev_days = {a + timedelta(days=i) for a, b in zip(ev.start_date, ev.end_date) for i in range((b - a).days + 1)}
on_event = (d.market_id.eq(spike_market) & d.style_id.eq(hero_style) & d.business_date.isin(ev_days)).to_numpy()

r = rng_for(seed, "ds11")
n = len(d)
exp, is_pre = d.exp_units.to_numpy(float), d.pre_launch.to_numpy()
mult = np.where(on_event, EVENT_MULT, 1.0) * engagement_scale
day_mk = d[["business_date", "market_id"]].drop_duplicates().reset_index(drop=True)
members = (loc[loc.location_type.isin(["STORE", "ECOM"])].groupby("market_id").size() * members_per_location)
weekend = pd.to_datetime(day_mk.business_date.astype(str)).dt.weekday.to_numpy() >= 5
day_mk["active_members"] = r.binomial(members.reindex(day_mk.market_id).fillna(0).astype(int).to_numpy(),
                                      np.where(weekend, 0.035 * 1.25, 0.035))
eng = d.merge(day_mk, on=["business_date", "market_id"])
eng = pd.DataFrame({
    "business_date": eng.business_date.astype(str), "market_id": eng.market_id, "style_id": eng.style_id,
    "color": eng.color, "active_members": eng.active_members,
    "app_product_opens": r.poisson(exp * 9.5 * mult * r.lognormal(0, 0.15, n)),
    "wishlist_adds": r.poisson(exp * np.where(is_pre, 2.2, 0.9) * mult),
    "launch_reminders": r.poisson(np.where(is_pre, exp * 1.4 * engagement_scale, 0.0)),
    "exported_at": utc_stamp(eng.business_date, 1, 3, 5),
    "record_label": SYNTHETIC})
if dq_rate > 0 and len(eng):
    k = max(1, int(r.binomial(len(eng), dq_rate)))
    dup = eng.iloc[r.choice(len(eng), k, replace=False)].copy()             # DQ-01: re-sent rows, later export wins
    dup["exported_at"] = utc_stamp(dup.business_date, 2, 3, 40)
    dup["wishlist_adds"] = dup.wishlist_adds + 1
    bad = eng.iloc[r.choice(len(eng), k, replace=False)].copy()             # DQ-02: unknown colour code
    bad["color"] = "999"
    eng = pd.concat([eng, dup, bad], ignore_index=True)

member_engagement = eng
save_bronze("member_engagement", member_engagement)

In [ ]:
# file_returns (DS-12): one return line per mirrored sell-through row with units_returned > 0
# Store returns carry local wall-clock time; e-commerce returns are UTC with a 'Z' suffix (DQ-08).
REASONS = {"SIZE_TOO_SMALL": (0.20, 0.26), "SIZE_TOO_LARGE": (0.14, 0.19), "COMFORT_FIT": (0.14, 0.10),   # (store, ecom)
           "DEFECTIVE": (0.08, 0.06), "NOT_AS_PICTURED": (0.04, 0.12), "CHANGED_MIND": (0.36, 0.20),
           "LATE_DELIVERY": (0.04, 0.07)}
codes = np.array(list(REASONS))
w_store, w_ecom = (np.array([v[i] for v in REASONS.values()]) for i in (0, 1))

x = s[s.units_returned > 0].reset_index(drop=True)
r = rng_for(seed, "ds12")
n = len(x)
ecom = x.location_type.eq("ECOM").to_numpy()
reason = np.where(ecom, r.choice(len(codes), n, p=w_ecom / w_ecom.sum()), r.choice(len(codes), n, p=w_store / w_store.sum()))
reason = np.where(x.style_id.eq(hero_style).to_numpy() & (r.random(n) < hero_small_return_share), 0, reason)
reason = np.where(~ecom & (codes[reason] == "LATE_DELIVERY"), list(codes).index("CHANGED_MIND"), reason)

units = x.units_returned.astype(int).to_numpy()
currency = x.market_id.map(markets.currency)
whole = currency.isin(["JPY", "KRW"]).to_numpy()
refund = price.reindex(pd.MultiIndex.from_arrays([x.market_id, x.style_id])).to_numpy() * units * np.where(r.random(n) < 0.25, 0.7, 1.0)
refund = np.where(whole, np.round(refund, 0), np.round(refund, 2))

day = pd.to_datetime(x.business_date.astype(str))
local_ts = day + pd.to_timedelta(r.integers(10 * 3600, 21 * 3600, n), unit="s")
return_ts = local_ts.dt.strftime("%Y-%m-%d %H:%M:%S").copy()
for zone, idx in x[ecom].groupby("timezone").groups.items():
    return_ts.loc[idx] = (local_ts.loc[idx].dt.tz_localize(zone, ambiguous="NaT", nonexistent="shift_forward")
                          .dt.tz_convert("UTC").dt.strftime("%Y-%m-%dT%H:%M:%SZ"))
ym = day.dt.strftime("%y%m")
seq = x.groupby([x.market_id, ym]).cumcount()

ret = pd.DataFrame({
    "return_id": "RMA-" + x.market_id + "-" + ym + "-" + seq.map("{:06d}".format), "return_line_no": "1",
    "return_ts": return_ts, "location_id": x.location_id, "channel": np.where(ecom, "ECOM", "STORE"),
    "sku_id": x.sku_id, "units": units, "reason_code": codes[reason],
    "item_condition": np.where(r.random(n) < 0.88, "RESELLABLE", "DAMAGED"), "refund_local": refund,
    "currency": currency, "original_purchase_date": (day - pd.to_timedelta(r.integers(2, 29, n), unit="D")).dt.strftime("%Y-%m-%d"),
    "exported_at": utc_stamp(x.business_date, 1, 4, 20), "record_label": SYNTHETIC})
if dq_rate > 0 and n:
    k = max(1, int(r.binomial(n, dq_rate)))
    dup = ret.iloc[r.choice(n, k, replace=False)].copy()                    # DQ-01: re-exported lines
    dup["exported_at"] = dup.exported_at.str.replace("T04:20", "T16:05")
    bad = ret.iloc[r.choice(n, k, replace=False)].copy()                    # DQ-02: unknown SKU
    bad["sku_id"] = bad.sku_id.str.rsplit("-", n=1).str[0] + "-99"
    ret = pd.concat([ret, dup, bad], ignore_index=True)

# DQ-05: bulk return booked at a slow-market store against the hero SKU it sold least of (returns > sales)
store = loc[(loc.market_id == slow_market) & (loc.location_type == "STORE")].sort_values("location_id").iloc[-1]
hero_skus = skus[skus.style_id == hero_style].sku_id.sort_values()
sold = s[s.location_id == store.location_id].groupby("sku_id").units_sold.sum().reindex(hero_skus).fillna(0)
bulk_sku, bulk_units = sold.idxmin(), int(max(24, sold.min() + 12))
ret = pd.concat([ret, pd.DataFrame([{
    "return_id": f"RMA-{slow_market}-BULK-000001", "return_line_no": "1", "return_ts": f"{end.isoformat()} 17:42:00",
    "location_id": store.location_id, "channel": "STORE", "sku_id": bulk_sku, "units": bulk_units,
    "reason_code": "DEFECTIVE", "item_condition": "DAMAGED",
    "refund_local": round(float(price[(slow_market, hero_style)]) * bulk_units, 2), "currency": markets.currency[slow_market],
    "original_purchase_date": (end - timedelta(days=3)).isoformat(),
    "exported_at": f"{as_of.isoformat()}T04:20:00Z", "record_label": SYNTHETIC}])], ignore_index=True)

returns = ret
save_bronze("returns", returns)

## Validate

In [ ]:
# Every table present, labelled SYNTHETIC, keyed to the mirrors and carrying its planted defects
expect = {"weather_daily", "regional_events", "member_engagement", "returns"}
assert expect <= set(generated), f"missing: {expect - set(generated)}"
for name, df in generated.items():
    assert len(df) > 0, f"{name}: no rows"
    assert (df.record_label == SYNTHETIC).all(), f"{name}: rows not labelled SYNTHETIC"
ge, gr, gw, gv = (generated[t] for t in ("member_engagement", "returns", "weather_daily", "regional_events"))
assert set(gr.location_id) <= set(loc.location_id), "returns: unknown location"
assert gr.sku_id[~gr.sku_id.str.endswith("-99")].isin(skus.sku_id).all(), "returns: unknown SKU outside the DQ-02 rows"
assert (gr.return_id == f"RMA-{slow_market}-BULK-000001").sum() == 1, "DQ-05 bulk return missing"
assert gw.temp_avg_c.isna().any(), "DQ-07 null temperatures missing"
assert (gw.groupby("market_id").size() >= 0.9 * ((end - start).days + 1)).all(), "weather: too many missing days"
assert ge.groupby("market_id").business_date.max().eq(end.isoformat()).all(), "engagement not delivered up to as_of - 1"
assert (gv.event_name == spike_event).any(), f"{spike_event} missing from the events feed"
if dq_rate > 0:
    assert (ge.color == "999").any(), "DQ-02 unknown colour missing"
    assert ge.duplicated(["business_date", "market_id", "style_id", "color"]).any(), "DQ-01 re-sent rows missing"
print({t: len(df) for t, df in generated.items()}, "- all checks passed")